# Aggregation

This notebook covers both classic workbook-based aggregation and workbook-free regional aggregation through `region_aggregation`.

In [ ]:
import mario
db = mario.load_test("IOT")

## Generate an empty Excel template

Open the generated workbook if you want to see the format MARIO expects.

In [ ]:
db.get_aggregation_excel(
    path="/path/to/empty_aggregation_template.xlsx",
    overwrite=True,
)

## Apply a filled Excel workbook

![The add-sectors cluster sheets](../../_static/images/aggregation_template.png)

The example workbook groups `Factors of production` as "Value added", and `Regions` into just one "World" region. The other levels are left blank, since they don't need to be filled if the user's intention is to leave them untouched.

### Download the packaged example workbooks

The exact workbooks used in this example are available here:

- [Empty aggregation template](../../_static/data/supporting_files/aggregation_iot_template.xlsx)
- [Filled aggregation workbook](../../_static/data/supporting_files/aggregation_iot_filled.xlsx)

In [ ]:
aggregated_db = db.aggregate(
    "/path/to/aggregation_iot_filled.xlsx",
    inplace=False,
    ignore_nan=True, # if you didn't aggregate all labels in all sets, this will ignore empty cells.
)

## Inspect the aggregated table

Compare aggregated and non-aggregated databases 

In [ ]:
aggregated_db

In [ ]:
db

Let's also obtain the Z matrix for the aggregated database and compare it with the one of the non-aggregated database

In [ ]:
aggregated_db.Z

In [ ]:
db.Z

## Workbook-free regional aggregation with known MRIOs

The same aggregation method can bypass the use of an Excel template if the user wants just to aggregate the `Region` level.
This is possible when using "MARIO-known" MRIO tables (e.g. EXIOBASE, EORA, OECD-ICIO...), thanks to an internal mapping process MARIO performs with built-in coverage rules, without asking you to fill the `Region` sheet manually.

The mapping is done in two steps. First, MARIO uses its packaged [Country_coverage workbook](../../_static/data/supporting_files/Country_coverage.xlsx) to reconcile source-specific region codes with ISO3 labels. Then, for labels that still need to be resolved, it falls back to [`country_converter`](https://github.com/IndEcol/country_converter), which provides the country metadata used to assign preset groups such as continents, UN regions, `EU`, `OECD`, `G7`, and `G20`.

For `region_aggregation`, the accepted preset strings are `"continent"`, `"UNregion"`, `"EU"`, `"OECD"`, `"G7"`, and `"G20"`. You can also pass an explicit mapping as a `dict`, `pandas.Series`, or `pandas.DataFrame` if you want full control over the target Region labels.

Below, a guided example using EXIOBASE. For monetary IOT tables, `mario.parse_exiobase(...)` accepts either the extracted dataset directory or the original `.zip` archive directly. With preset region aggregation, the resulting Region labels are plain names such as `Europe`, `Africa`, `Asia`, `Oceania`, and `America`; the `continent:` prefix is kept only for reusable default cluster names.

In [14]:
import mario

exio_db = mario.parse_exiobase(
    path="/path/to/IOT_2024_ixi.zip",  # or the extracted dataset directory
    table="IOT",
    unit="Monetary",
)

exio_db.regions  # print regions

['AT',
 'BE',
 'BG',
 'CY',
 'CZ',
 'DE',
 'DK',
 'EE',
 'ES',
 'FI',
 'FR',
 'GR',
 'HR',
 'HU',
 'IE',
 'IT',
 'LT',
 'LU',
 'LV',
 'MT',
 'NL',
 'PL',
 'PT',
 'RO',
 'SE',
 'SI',
 'SK',
 'GB',
 'US',
 'JP',
 'CN',
 'CA',
 'KR',
 'BR',
 'IN',
 'MX',
 'RU',
 'AU',
 'CH',
 'TR',
 'TW',
 'NO',
 'ID',
 'ZA',
 'WA',
 'WL',
 'WE',
 'WF',
 'WM']

In [ ]:
exio_by_continent = exio_db.aggregate(
    io=None,
    levels="Region",
    region_aggregation="continent",
    inplace=False,
)

exio_by_continent.regions